# Correlation and Normality

**Thursday · Data Analytics**

**Learning goals:** interpret correlation coefficients, connect Pearson correlation to a hypothesis test, inspect normality with histograms and Q-Q plots, understand Shapiro–Wilk, and evaluate transformations.

We will use a housing-price dataset. The focus is always **question → assumptions → statistic → p-value → interpretation**, not memorizing a list of tests.

## 0. Setup and data

We use `SalePrice`, `LotArea`, and `1stFlrSF`. The data come from a public housing dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
alpha = 0.05
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/housing_price_eda.csv"
df = pd.read_csv(url)
columns = ["SalePrice", "LotArea", "1stFlrSF"]
print(df[columns].describe().round(2))

## 1. Correlation: what are we measuring?

A correlation coefficient measures the **direction and strength of association**, not a causal effect. Pearson’s coefficient measures **linear** association and ranges from −1 to +1.

- Positive: larger values of one variable tend to accompany larger values of the other.
- Negative: larger values tend to accompany smaller values.
- Near zero: little *linear* association; nonlinear relationships may still exist.

A scatterplot should come **before** the coefficient.

In [ ]:
# Inspect the scatterplots before calculating Pearson correlation.
# Highlight the three observations with the largest absolute regression residuals
# as candidates for closer inspection, NOT as automatically invalid records.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col in zip(axes, ["LotArea", "1stFlrSF"]):
    paired = df[[col, "SalePrice"]].dropna()
    x = paired[col].to_numpy(dtype=float)
    y = paired["SalePrice"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(x, y, deg=1)
    residuals = y - (slope * x + intercept)
    candidate_positions = np.argsort(np.abs(residuals))[-3:]
    candidates = np.zeros(len(x), dtype=bool)
    candidates[candidate_positions] = True

    ax.scatter(x[~candidates], y[~candidates], s=18, alpha=0.45,
               color="steelblue", label="Other observations")
    ax.scatter(x[candidates], y[candidates], s=60, color="darkorange",
               edgecolors="black", linewidths=0.6, zorder=3,
               label="Largest residuals: inspect")
    ax.set(xlabel=col, ylabel="SalePrice", title=f"{col} vs SalePrice")
    ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

**Why are some points orange?** These are the three observations with the largest *absolute vertical residuals* from a fitted straight line in each plot. They are **candidates for inspection**, not automatically errors or observations to remove. This is a simple illustration of a *bivariate* concern: a point may have an ordinary value on each individual axis but an unusual combination of the two values.

The residual criterion depends on the fitted line and can be distorted by influential observations. More formal approaches include standardized or studentized residuals, leverage, Cook’s distance, and robust multivariate methods. We will not introduce those procedures here.

**Before computing Pearson's r:** inspect the highlighted records, check for data-entry errors and consider whether a few influential points change the interpretation. Do not delete genuine observations merely to increase correlation or obtain a desired p-value.

### Pearson correlation: the formula

$$
r=\frac{\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_{i=1}^{n}(x_i-\bar x)^2}\sqrt{\sum_{i=1}^{n}(y_i-\bar y)^2}}
$$

Here, $x_i,y_i$ are paired observations, $\bar x,\bar y$ are their sample means, and $n$ is the number of pairs. The numerator measures how the variables vary together; the denominator standardizes that quantity.

**Important:** Pearson correlation can be affected by outliers, and a small $r$ does not prove independence.

In [ ]:
for col in ["LotArea", "1stFlrSF"]:
    paired = df[[col, "SalePrice"]].dropna()
    r = paired[col].corr(paired["SalePrice"])
    print(f"Pearson r ({col}, SalePrice) = {r:.3f}")

### From correlation to a hypothesis test

**Question:** Is there evidence of a nonzero *population* linear correlation?

$$H_0:\rho=0 \qquad H_1:\rho\ne0$$

The **alternative** expresses what we want evidence for. This is a two-sided test because both positive and negative correlations contradict the null.

For independent pairs from a bivariate normal population, under $H_0$:

$$t_{\mathrm{obs}}=r\sqrt{\frac{n-2}{1-r^2}},\qquad df=n-2$$

For the two-sided p-value: $\mathrm{p\text{-}value}=2\,\mathrm{SF}_{t_{n-2}}(|t_{\mathrm{obs}}|)$.

In [ ]:
paired = df[["LotArea", "SalePrice"]].dropna()
n_pairs = len(paired)
r = paired["LotArea"].corr(paired["SalePrice"])
t_obs = r * np.sqrt((n_pairs - 2) / (1 - r**2))
p_value = 2 * stats.t.sf(abs(t_obs), df=n_pairs - 2)
scipy_result = stats.pearsonr(paired["LotArea"], paired["SalePrice"])
print(f"r = {r:.3f}; t = {t_obs:.3f}; p-value = {p_value:.3f}")
print(f"SciPy: r = {scipy_result.statistic:.3f}; p-value = {scipy_result.pvalue:.3f}")
print("Reject H0" if p_value <= alpha else "Do not reject H0")

**Interpretation:** a small p-value supports evidence of nonzero population linear association. It does **not** tell us whether the relationship is strong or useful for prediction; for that we examine $r$ and the scatterplot. Nor does it establish causation.

### Check for understanding (3 minutes)

A dataset produces $r=0.08$ and `p-value = 0.002`. Is the relationship statistically detectable? Is it strong? Explain why these are different questions.

### Choosing an association measure: Pearson or Spearman

Before selecting a coefficient, ask **what kind of relationship are we investigating?**

| Variables and relationship | Useful measure | What it describes |
|---|---|---|
| Two numerical variables with an approximately linear relationship | **Pearson’s $r$** | Strength and direction of linear association |
| Two numerical or ordinal variables with a monotonic relationship | **Spearman’s $\rho_s$** | Strength and direction of rank-based, monotonic association |

**Remember:** a monotonic relationship consistently increases or decreases, but it need not follow a straight line. Always inspect the scatterplot before choosing a coefficient.

### 💡 Check for understanding (3 minutes)

For each question, choose **Pearson or Spearman**, and explain why:

1. Is there a linear relationship between a house's floor area and its sale price?
2. Do customers with higher satisfaction rankings tend to give higher recommendation rankings?
3. Two numerical variables follow a steadily increasing but curved pattern. Which coefficient better captures the association?

**Discuss:** Can Pearson's $r$ be close to zero even if there is a strong relationship between two variables?

## 2. What does “normally distributed” mean?

A normal distribution is characterized by its mean $\mu$ and standard deviation $\sigma$. Real data do not have to be perfectly bell-shaped, and **not every statistical test assumes that the raw data are normal**.

We will examine each variable in two ways: **histogram** (overall shape) and **Q-Q plot** (observed quantiles versus theoretical normal quantiles).

In [ ]:
def normality_diagnostics(values, title):
    values = pd.Series(values).dropna().to_numpy(dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(values, bins=30, kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title(f"{title}: histogram")
    stats.probplot(values, dist="norm", plot=axes[1])
    axes[1].set_title(f"{title}: normal Q-Q plot")
    plt.tight_layout()
    plt.show()

for col in columns:
    normality_diagnostics(df[col], col)

### Reading the Q-Q plot

If the observations were approximately normal, points would lie roughly along the reference line. Systematic curvature suggests skewness or a mismatch in the tails. A few extreme points can indicate outliers or heavier tails.

**Check for understanding (3 minutes):** Which of the three variables seems most skewed? What evidence do you see in *both* plots?

## 3. Shapiro–Wilk: a hypothesis test for normality

$$H_0:\text{The population distribution is normal}$$
$$H_1:\text{The population distribution is not normal}$$

The test statistic $W$ measures how well the ordered observations agree with the pattern expected from a normal sample. Values closer to 1 generally indicate better agreement. SciPy calculates the statistic and its p-value.

**Decision rule:** choose $\alpha$ **before** examining results. Reject $H_0$ if `p-value <= alpha`; otherwise, do not reject it. Failure to reject does **not** prove normality.

**Caution:** with large samples, small deviations may be statistically significant even when they have little practical impact. Combine the test with the plots. For samples larger than 5,000, SciPy warns that its Shapiro p-value may be inaccurate.

In [ ]:
for col in columns:
    values = df[col].dropna().to_numpy()
    w, p_value = stats.shapiro(values)
    decision = "Reject H0" if p_value <= alpha else "Do not reject H0"
    print(f"{col:10s} | W = {w:.3f} | p-value = {p_value:.3f} | {decision}")

### Another normality test: Kolmogorov–Smirnov (K–S)

The **one-sample Kolmogorov–Smirnov test** compares the empirical cumulative distribution function (ECDF) of a sample with a fully specified theoretical CDF. Its statistic is the largest vertical distance between the two curves:

$$D=\sup_x\left|F_n(x)-F_0(x)\right|$$

Here, $F_n(x)$ is the sample ECDF and $F_0(x)$ is the theoretical CDF. For a normality check, $F_0$ would be a normal CDF with **specified** population parameters.

**Why not use the ordinary K–S test here?**

If we standardize a variable using its **sample mean and sample standard deviation**, and then call `kstest(..., "norm")`, the usual K–S p-value is not valid: the reference distribution has been fitted using the same observations.

When the normal mean and standard deviation are **estimated from the same sample**, the usual K–S reference distribution no longer applies. The **Lilliefors correction/test** accounts for estimating those parameters. A naive `stats.kstest` call after sample standardization does **not** apply that correction automatically.

For this lesson, use **Shapiro–Wilk together with graphical diagnostics**. We mention K–S and Lilliefors for context, without adding another test to memorize.

### How large can the sample be for Shapiro–Wilk?

There is **no single sample size** above which normality tests suddenly become useless.

- For **more than 5,000 observations**, SciPy warns that Shapiro–Wilk's **W statistic may be accurate, but its p-value may not be**.
- Even for smaller samples, a **very large n** gives the test high power: it may reject normality for tiny deviations that are not practically important.
- For **small n**, failing to reject normality does not establish that the population is normal; the test may have little power.

**Practical rule:** always inspect the histogram and Q–Q plot, and ask whether the departure from normality matters for the intended analysis. A small p-value is evidence against exact normality, not a measure of how severe the deviation is.

**Check for understanding (3 minutes):** We draw a random sample of 40 observations from a population and inspect its histogram and Q–Q plot. Then we draw another independent sample of 40 observations from the same population. Could the plots and Shapiro–Wilk statistic change noticeably? Why? What would you expect if both samples contained 4,000 observations instead?

**Discussion:** Small samples have greater sampling variability, so their plots and $W$ can fluctuate substantially. Large samples generally describe the population shape more consistently, but Shapiro–Wilk also has greater power to detect even minor departures from normality. A stable-looking distribution and a statistically significant test are not contradictory.

## 4. Transformations: what exactly are we changing?

There are **two useful categories**:

1. **Scale and location transformations** change the units, spread or center, but **do not change the shape** of the distribution. For example, $Y=aX+b$ (with $a>0$): converting euros to thousands of euros or standardizing to a z-score. Think of *zooming in/out* or shifting the horizontal axis.
2. **Shape transformations** change the spacing between observations nonlinearly and can change skewness. Examples include logarithms, square roots and Box–Cox.

The **z-score standardization** of an observation is

$$
z_i=\frac{x_i-\bar{x}}{s}
$$

where $x_i$ is an observation, $\bar{x}$ is the sample mean and $s$ is the **sample standard deviation** (not the population standard deviation $\sigma$). The resulting sample has mean 0 and sample standard deviation 1, provided $s>0$.

**Important:** *standardizing a skewed variable* means applying this z-score transformation; it does **not** make the variable normal. And making a variable look more symmetric is not always necessary or desirable.

### A. Scale and location: same shape, different axis

We will compare `SalePrice` with `SalePrice / 1000` and its standardized version. The histograms should have the same shape even though the numerical axes differ.

### B. Shape: logarithm and square root

- `np.log1p(x)` means **$\log(1+x)$**, where $\log$ is the natural logarithm. It is particularly convenient when **$x=0$** is possible, since $\log(0)$ is undefined. Its mathematical domain is **$x>-1$**; for ordinary nonnegative counts or amounts, $x\geq0$ is the usual use case.
- `np.log(x)` requires **$x>0$**.
- `np.sqrt(x)` requires **$x\geq0$** for real-valued outputs.

These nonlinear transformations can reduce right skewness, but they also change the meaning of distances and differences.

### C. Box–Cox: a flexible family of power transformations

For a **strictly positive** variable ($x>0$), Box–Cox searches for a power parameter $\lambda$ that makes the transformed data closer to a normal distribution:

$$
T_\lambda(x)=
\begin{cases}
\dfrac{x^\lambda-1}{\lambda}, & \lambda\neq0\\
\log(x), & \lambda=0
\end{cases}
$$

The offset and division are part of the definition; Box–Cox is not simply $x^\lambda$. `scipy.stats.boxcox` estimates $\lambda$ by maximizing a likelihood under a normality model. It does **not guarantee** normality.

**Beyond this lesson:** `scipy.stats.yeojohnson` is a related power transformation that also accepts zero and negative values. We will not use it in the exercises.

In [ ]:
saleprice = df["SalePrice"].dropna()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, values, title in zip(
    axes,
    [saleprice, saleprice / 1000, (saleprice - saleprice.mean()) / saleprice.std(ddof=1)],
    ["SalePrice ($)", "SalePrice (thousands of $)", "Standardized SalePrice (z-score)"],
):
    ax.hist(values, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(title)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Comparing shape transformations on the **same variable**

We will apply all transformations to **`SalePrice`**, so every histogram and Q–Q plot refers to the same observations. This lets us compare the effect of each mathematical operation rather than differences between variables.

- **Original:** unchanged observations.
- **Z-score:** changes location and scale, but preserves shape.
- **`log1p(x)`**: $\log(1+x)$; requires $x>-1$ (typically used for nonnegative data).
- **Square root:** requires $x\geq0$.
- **Box–Cox:** requires **strictly positive** values and estimates $\lambda$ from the sample.
- **Yeo–Johnson (optional):** accepts negative and zero values as well; available in SciPy as `stats.yeojohnson`.

The histograms use the same number of bins, but their horizontal units differ. The Q–Q plots allow us to assess the shape independently of location and scale. **A transformation need not make the data perfectly normal to be useful.**

In [ ]:
# Compare the same observations before and after each transformation.
# All transformations are computed from the same non-missing SalePrice values.
values = df["SalePrice"].dropna().to_numpy(dtype=float)
assert np.isfinite(values).all(), "SalePrice must contain finite observations."
assert (values > 0).all(), "Box-Cox requires strictly positive observations."

z_scores = (values - values.mean()) / values.std(ddof=1)
boxcox_values, boxcox_lambda = stats.boxcox(values)
yeojohnson_values, yeojohnson_lambda = stats.yeojohnson(values)

transformations = [
    ("Original", values),
    ("Z-score", z_scores),
    ("log1p", np.log1p(values)),
    ("Square root", np.sqrt(values)),
    (f"Box-Cox (λ={boxcox_lambda:.3f})", boxcox_values),
    (f"Yeo-Johnson (λ={yeojohnson_lambda:.3f})", yeojohnson_values),
]

fig, axes = plt.subplots(
    len(transformations), 2,
    figsize=(12, 3.1 * len(transformations)),
    constrained_layout=True,
)

for row, (name, transformed) in enumerate(transformations):
    ax_hist, ax_qq = axes[row]
    ax_hist.hist(transformed, bins=30, color="steelblue", edgecolor="white")
    ax_hist.set_title(f"{name} — histogram")
    ax_hist.set_ylabel("Count")
    ax_hist.set_xlabel("Transformed value")

    stats.probplot(transformed, dist="norm", plot=ax_qq)
    ax_qq.set_title(f"{name} — normal Q-Q plot")
    ax_qq.get_lines()[0].set_markerfacecolor("steelblue")
    ax_qq.get_lines()[0].set_markeredgecolor("steelblue")

plt.show()

print(f"Box-Cox lambda = {boxcox_lambda:.3f}")
print(f"Yeo-Johnson lambda = {yeojohnson_lambda:.3f}")

In [ ]:
# Compare Shapiro-Wilk across the transformations of the same observations.
# A larger p-value is NOT a goal in itself: interpret the plots and the use case.
for name, transformed in transformations:
    w, p_value = stats.shapiro(transformed)
    print(f"{name:28s} | W = {w:.3f} | p-value = {p_value:.3f}")

In [ ]:
# A separate example: Box-Cox applied to LotArea (strictly positive).
lot_area = df["LotArea"].dropna().to_numpy(dtype=float)
assert (lot_area > 0).all(), "Box-Cox requires strictly positive observations."
lot_boxcox, fitted_lambda = stats.boxcox(lot_area)
print(f"LotArea Box-Cox lambda = {fitted_lambda:.3f}")
normality_diagnostics(lot_boxcox, "Box-Cox(LotArea)")

### Interpretation after transformation

Compare the histogram, Q–Q plot and Shapiro–Wilk statistic **before and after**, using the same observations. A higher p-value is not an optimization target: transformations should be chosen for a clear analytical purpose and interpreted in their transformed units.

**Check for understanding (4 minutes):** Which operations changed only the axis? Which changed the shape? Why can we use `log1p` on a zero but not `log`? Why does Box–Cox require strictly positive observations?

## 5. Practical decision guide

| Question | First look at | Then calculate | Key caution |
|---|---|---|---|
| Are two numeric variables linearly associated? | Scatterplot | Pearson $r$ and its test | Correlation is not causation |
| Are two variables monotonically associated (including ranks)? | Scatterplot or rank plot | Spearman $\rho_s$ | Monotonic does not necessarily mean linear |
| Does a variable look approximately normal? | Histogram and Q-Q plot | Shapiro–Wilk | Large $n$ can detect tiny departures |
| Does a transformation help? | Before/after plots | Compare diagnostics | Interpretation changes with scale |

**Remember:** statistical significance is not the same as practical importance. The assumptions concern the *particular method*, and they do not always require the original variables to be normal.

## Final recap (5 minutes)

1. Why can $r$ be small even when its p-value is very small?
2. What does a curved Q-Q plot suggest?
3. If Shapiro–Wilk returns `p-value = 0.20`, have we proved the population is normal?
4. What changes when we replace prices with their logarithms?

**Takeaway:** formulate the question, visualize the data, understand the statistic and its assumptions, compute the p-value, and interpret the result in context.